# Results 00 · Inventory

What results exist, whether they are complete, and whether all inputs were evaluated on the same samples. Run this first; the other notebooks plot whatever is listed here as complete.

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import results_viz as rv

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 200)

# Settings
INCLUDE_PARTIAL = True  # the inventory shows runs in progress too   # True: also show runs still in progress (marked in the inventory)
SAVE_FIGURES = True       # PNG + PDF in notebooks/figures/results/

df = rv.load_all(include_partial=INCLUDE_PARTIAL)
pp = rv.per_problem(df) if not df.empty else df
print(f"{len(df)} result rows, {len(pp)} problems x feature sets; grids: {sorted(df['grid'].unique()) if len(df) else []}")
print("thresholds found:", rv.thresholds(df) if len(df) else [])


## Coverage

Number of problems per feature set and scheme. `(partial)`: the run is still in progress or was cut off. `—`: no results.

In [ ]:
cov = rv.coverage(df)
cov

## Sample-count consistency

Every input is evaluated on the same problems; a problem whose number of samples differs between inputs would indicate misaligned data. **This table should be empty.**

In [ ]:
complete = df[df['run_status'] == 'complete'] if 'run_status' in df else df
rv.sample_count_check(complete)

## Runs

Run folder, date and code version of each configuration used.

In [ ]:
import json
from pathlib import Path

def run_info(result_dir):
    cfg = json.loads((Path(result_dir) / "run_config.json").read_text())
    return pd.Series({"timestamp": cfg.get("timestamp"), "git_commit": (cfg.get("git_commit") or "")[:12],
                      "status": cfg.get("status", "complete")})

runs = df.drop_duplicates("result_dir")[["grid", "label", "target", "scope", "run_dir", "result_dir"]]
runs = pd.concat([runs, runs["result_dir"].apply(run_info)], axis=1)
runs["run"] = runs["run_dir"].map(lambda p: Path(p).name)
runs.groupby(["grid", "label", "run", "git_commit", "status"]).size().rename("result folders").to_frame()